In [1]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)


from vllm_wrapper import VLLMRaterModel
from eval.safe.rate_atomic_fact import check_atomic_fact

2025-12-11 18:17:50.073955931 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Initializing BM25 Index... (This might take a moment)


/root/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Dec 11, 2025 6:17:50 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


BM25 Index loaded successfully.


In [6]:
import json
atomic_fact_data = []
# In Jupyter notebooks, use getcwd() instead of __file__
path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))

completed_facts = []
path_to_completed = os.path.join(os.getcwd(), "data_for_git", "rated_facts.jsonl")
with open(path_to_completed, "r", encoding="utf-8") as f:
    for line in f:
        completed_facts.append(json.loads(line))

print("number of completed facts: ", len(completed_facts))
print(completed_facts[0]["id"])


# check how many unique ids are in atomic_fact_data
atomic_ids = [fact["id"] for fact in atomic_fact_data]
atomic_unique_ids = set(atomic_ids)
print("number of unique ids in atomic_fact_data: ", len(atomic_unique_ids))



# Check for duplicate IDs in completed_facts
ids = [fact["id"] for fact in completed_facts]
unique_ids = set(ids)
duplicate_ids = [id for id in unique_ids if ids.count(id) > 1]

print(len(duplicate_ids))

# filter atomic_fact_data to only include facts that are not in completed_facts
# Count total unique facts (similar to facts_completed but without duplicates)
# Convert dicts to JSON strings for hashing (since dicts are not hashable)
unique_completed_facts = len(set(json.dumps(fact, sort_keys=True) for query in completed_facts for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]))
print("unique_completed_facts: ", unique_completed_facts)
before = len(atomic_fact_data)
facts_before = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
completed_ids_set = set(ids)  # Use the IDs we already extracted above
atomic_fact_data = [fact for fact in atomic_fact_data if fact["id"] not in completed_ids_set]
after = len(atomic_fact_data)
filtered_out = before - after
print(f"Before: {before}, After: {after}, Filtered out: {filtered_out}")


number of completed facts:  3512
be3ff6b1-3f3e-4d3c-bcef-62473da8c672
number of unique ids in atomic_fact_data:  3413
85
unique_completed_facts:  112719
Before: 3413, After: 0, Filtered out: 3413


In [3]:
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
print(total_facts)
print(len(atomic_fact_data))
atomic_fact_data[0]["results"]["all_atomic_facts"][0]["atomic_facts"]

facts_completed = len([fact for query in completed_facts for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])

print(unique_completed_facts)
print(facts_completed)
print(total_facts)
print(facts_before)

17835
318
97108
107277
17835
124643


In [4]:
from pydantic import BaseModel
from openai import OpenAI
class Response(BaseModel):
    answer: str

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
structured_output = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that is either tasked with finding a good search query or deciding if a fact is supported or not. You will be given a prompt and a response format. Pay close attention to the response format and do not deviate from it."},
                    {"role": "user", "content": "Hello, how are you?"}
                ],
                # extra_body={
                #     "guided_json": response_format.model_json_schema() },
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "example",
                        "schema": Response.model_json_schema()
                    }
                }
)
print(structured_output)

ChatCompletion(id='chatcmpl-adff9871ab786432', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"answer":"I’m doing well, thanks for asking! How can I help you today?"}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='The system says: "You are a helpful assistant that is either tasked with finding a good search query or deciding if a fact is supported or not. You will be given a prompt and a response format. Pay close attention to the response format and do not deviate from it." The user says: "Hello, how are you?" There\'s no prompt about searching or fact checking. The user just greets. The instruction requires that we respond in a specific format based on the task. But the user hasn\'t given a prompt. Probably it\'s a trick: The user is casual; we are supposed to respond politely. But the system sets that we have to use the response format depending on context. There\'s 

In [5]:
llm = VLLMRaterModel()
import threading
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json


results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
pbar_lock = threading.Lock()

MAX_WORKERS = 400          # outer: responses
MAX_FACT_WORKERS = 4       # inner: facts per response (tune)

def rate_fact(fact, bio_person, llm, idx):
    rating = check_atomic_fact(fact, bio_person, llm, max_steps=2)[0].answer
    return idx, rating

def worker(full_dict, pbar):
    try:
        bio_person = full_dict["prompt"].split("Tell me a bio of ")[1]
        all_atomic_facts = full_dict["results"]["all_atomic_facts"]

        # Collect all facts to rate with their (sf_idx, fact_idx) positions
        jobs = []
        for sf_idx, sentence_facts in enumerate(all_atomic_facts):
            for fact_idx, fact in enumerate(sentence_facts["atomic_facts"]):
                jobs.append((sf_idx, fact_idx, fact))

        # Rate facts in parallel within this response
        with ThreadPoolExecutor(max_workers=MAX_FACT_WORKERS) as fact_pool:
            futures = [
                fact_pool.submit(rate_fact, fact, bio_person, llm, (sf_idx, fact_idx))
                for sf_idx, fact_idx, fact in jobs
            ]
            for fut in as_completed(futures):
                (sf_idx, fact_idx), rating = fut.result()
                all_atomic_facts[sf_idx]["atomic_facts"][fact_idx] = {
                    "fact": all_atomic_facts[sf_idx]["atomic_facts"][fact_idx],
                    "rating": rating,
                }
                with pbar_lock:
                    pbar.update(1)

        return ("success", full_dict)
    except Exception as e:
        return ("error", f"Error processing response {full_dict.get('id', '?')}: {e}")

first_n = None
facts_completed = len([fact for query in completed_facts for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
current_progress = len(completed_facts)

# Define output file for incremental saving
output_file = os.path.join(os.getcwd(), "data_for_git", "rated_facts.jsonl")

# 2. Run wth Executor
tasks = atomic_fact_data
# Create both progress bars with position parameter to display them separately
with tqdm(total=facts_before, desc="Rating Facts", position=0, leave=True) as task_bar:
    task_bar.update(unique_completed_facts)
    with tqdm(total=before, desc="Completed Responses", position=1, leave=True) as pbar:
        pbar.update(filtered_out)
        tqdm.write(f"Starting from {filtered_out}/{before} ({filtered_out/before * 100}%)")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks
            futures = [executor.submit(worker, task, task_bar) for task in tasks]
            
            # Process results as they complete
            for future in as_completed(futures):
                status, payload = future.result()
                
                if status == "success":
                    # Save to disk immediately (append mode)
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(json.dumps(payload) + "\n")
                    pbar.update(1)
                    
                    with results_lock:
                        results.append(payload)
                else:
                    with error_log_lock:
                        error_log.append(payload)

# 3. Save Logs
with open(os.getcwd() + "/data_for_git/fact_rating_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

print(f"Processed {len(results)} responses successfully.")

                                                        
Rating Facts:  78%|███████▊  | 97108/124643 [00:00<00:00, 76202146.46it/s]

Starting from 3095/3413 (90.6826838558453%)


Rating Facts:  92%|█████████▏| 114943/124643 [2:14:55<11:23, 14.20it/s]    

Processed 318 responses successfully.


In [23]:
#write results to file with utf8 encoding
with open(os.getcwd() + "/data_for_git/rated_facts.jsonl", "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")
